## 연습 1: AutoSphere 데이터 플랫폼 구성

당신은 글로벌 자동차 제조사 **AutoSphere AG**의 데이터 엔지니어입니다. 이 회사의 커넥티드 차량 플랫폼은 수백만 대 차량의 텔레메트리를 수집하고, 고객 등록 데이터를 관리하며, 서비스 이력을 추적합니다.

거버넌스 제어를 적용하기 전에, 먼저 플랫폼의 기반이 되는 카탈로그, 스키마, 테이블을 생성해야 합니다. 또한 각 테이블과 컬럼의 목적을 문서화하기 위해 **주석(comment)**을 추가하여, 다른 팀이 데이터 자산을 쉽게 발견할 수 있도록 합니다.

### 설정 — 카탈로그, 스키마, 테이블 생성

거버넌스 제어에 집중할 수 있도록 AutoSphere 데이터 플랫폼은 아래 셀에서 미리 구성되어 있습니다. 이 셀은 다음을 생성합니다:

- **`automotive_catalog`** — AutoSphere 플랫폼을 위한 Unity Catalog 카탈로그(설명 주석 포함)
- **`automotive_catalog.governance_lab`** — 이 실습 전반에서 사용하는 스키마
- **`customer_registrations`** — 등록된 고객 및 해당 차량당 한 행씩 저장; 테이블 수준 주석 포함
- **`vehicle_telemetry`** — 커넥티드 차량의 연속 텔레메트리 이벤트; `speed_kmh`와 `battery_level_pct`에 컬럼 수준 주석 포함

셀을 실행하여 환경을 준비한 뒤, 연습 2로 진행하세요.

In [ ]:
# 기본 스토리지가 활성화되어 있지 않으므로, 기본 카탈로그의 루트에서 스토리지 경로를 상속받습니다.
catalogs = [catalog.name for catalog in spark.catalog.listCatalogs("adb_dp*")]
dp750_default_catalog = catalogs[0] if catalogs else None

storage_root = (
    spark.sql(f"DESCRIBE CATALOG EXTENDED {dp750_default_catalog}")
    .filter("info_name = 'Storage Root'")
    .select("info_value")
    .first()[0]
)
print (f"Storage root: {storage_root}")

spark.sql(f"""
    CREATE CATALOG IF NOT EXISTS automotive_catalog
    MANAGED LOCATION '{storage_root}'
    COMMENT 'AutoSphere AG의 커넥티드 차량 플랫폼 카탈로그 — 등록, 텔레메트리, 서비스 데이터를 저장합니다.'
""")

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS automotive_catalog.governance_lab
  COMMENT 'Unity Catalog 실습 연습을 위한 거버넌스 실습 스키마.';

-- customer_registrations 테이블 생성
CREATE TABLE IF NOT EXISTS automotive_catalog.governance_lab.customer_registrations
  COMMENT '등록된 AutoSphere 고객 및 해당 차량당 한 행.'
AS SELECT * FROM VALUES
  (1, 'Lukas Bauer',    'lukas.bauer@autosphere.de',   'DE-LB-4821', 'DE', 'VH-001'),
  (2, 'Sophie Martin',  'sophie.martin@autosphere.fr',  'FR-SM-9034', 'FR', 'VH-002'),
  (3, 'James Clarke',   'james.clarke@autosphere.uk',   'UK-JC-1172', 'UK', 'VH-003'),
  (4, 'Yuki Tanaka',    'yuki.tanaka@autosphere.jp',    'JP-YT-5561', 'JP', 'VH-004'),
  (5, 'Maria Santos',   'maria.santos@autosphere.pt',   'PT-MS-8843', 'PT', 'VH-005'),
  (6, 'Carlos Rivera',  'carlos.rivera@autosphere.es',  'ES-CR-3309', 'ES', 'VH-006')
AS t(customer_id, full_name, email, driver_license_no, country, vehicle_id);

-- vehicle_telemetry 테이블 생성
CREATE TABLE IF NOT EXISTS automotive_catalog.governance_lab.vehicle_telemetry (
  event_id          BIGINT    COMMENT '고유 텔레메트리 이벤트 식별자',
  vehicle_id        STRING,
  event_time        TIMESTAMP,
  speed_kmh         INT       COMMENT '순간 차량 속도(시속 킬로미터)',
  battery_level_pct INT       COMMENT '충전 상태 백분율(0–100)',
  latitude          DOUBLE,
  longitude         DOUBLE,
  country           STRING
);

INSERT INTO automotive_catalog.governance_lab.vehicle_telemetry VALUES
  (1, 'VH-001', TIMESTAMP '2026-03-01 08:15:00', 112, 78,  48.8566,   2.3522, 'DE'),
  (2, 'VH-002', TIMESTAMP '2026-03-01 08:16:00', 95,  55,  51.5074,  -0.1278, 'FR'),
  (3, 'VH-003', TIMESTAMP '2026-03-01 08:17:00', 130, 91,  40.4168,  -3.7038, 'UK'),
  (4, 'VH-004', TIMESTAMP '2026-03-01 08:18:00', 88,  34,  35.6762, 139.6503, 'JP'),
  (5, 'VH-005', TIMESTAMP '2026-03-01 08:19:00', 105, 62,  38.7223,  -9.1393, 'PT'),
  (6, 'VH-006', TIMESTAMP '2026-03-01 08:20:00', 77,  47,  41.3851,   2.1734, 'ES');

## 연습 2: 거버넌스를 위한 데이터 자산 태깅

AutoSphere의 컴플라이언스 팀은 **개인 식별 정보(PII)**를 포함하는 모든 테이블과 컬럼에 일관된 레이블을 지정하도록 요구합니다. 태그는 자동 검색을 가능하게 하며, 이후 접근 정책을 구동하는 데 사용됩니다.

이 연습에서는 `customer_registrations` 테이블을 분류하기 위해 테이블 수준 및 컬럼 수준 태그를 적용합니다.

### 작업 2.1 — 테이블 수준 태그 추가

`customer_registrations` 테이블을 도메인과 민감도로 분류하도록 태그를 지정하세요:

- `domain` = `customer`
- `data_classification` = `confidential`

> 🤖 **Genie Code 팁:** 다음과 같이 질문해 보세요:
> *"Databricks SQL에서 Unity Catalog 테이블에 키-값 태그를 어떻게 추가하나요?"*

**힌트:** `ALTER TABLE ... SET TAGS ('key' = 'value', 'key2' = 'value2')`를 사용하세요.

In [ ]:
%sql
-- TODO: automotive_catalog.governance_lab.customer_registrations에 테이블 수준 태그 추가
-- 태그: domain = customer, data_classification = confidential

### 작업 2.2 — 컬럼 수준 PII 태그 추가

`customer_registrations`의 PII 컬럼을 민감 정보로 식별하도록 태그를 지정하세요:

- `email` 컬럼 → `pii` = `email`
- `driver_license_no` 컬럼 → `pii` = `driver_license`

> 🤖 **Genie Code 팁:** 다음과 같이 질문해 보세요:
> *"Databricks에서 SQL을 사용해 Unity Catalog 테이블의 특정 컬럼에 태그를 어떻게 추가하나요?"*

**힌트:** `ALTER TABLE ... ALTER COLUMN <col> SET TAGS (...)`를 사용하세요.

In [ ]:
%sql
-- TODO: email 컬럼에 pii = email 태그 지정

-- TODO: driver_license_no 컬럼에 pii = driver_license 태그 지정

### 작업 2.3 — 태그 확인

`system.information_schema.table_tags`를 사용하여 테이블 수준 태그가 적용되었는지 확인하세요. 그런 다음 컬럼을 설명하여 컬럼 태그가 존재하는지 확인하세요. 이를 위해 `system.information_schema.column_tags`를 사용하세요.

> 🤖 **Genie Code 팁:** 다음과 같이 질문해 보세요:
> *"Databricks SQL에서 Unity Catalog 테이블과 컬럼의 태그를 어떻게 확인하나요?"*

In [ ]:
%sql
-- customer_registrations의 테이블 수준 태그 표시

In [ ]:
%sql
-- customer_registrations의 컬럼 수준 태그 표시

## 연습 3: 데이터 보존 구성

AutoSphere의 차량 텔레메트리 테이블에는 데이터가 지속적으로 기록됩니다. GDPR 데이터 최소화 원칙을 준수하고 스토리지 비용을 제어하기 위해 플랫폼은 다음을 수행해야 합니다:

- 테이블 이력을 **14일**만 보존합니다.
- `VACUUM`을 사용하여 오래된 데이터 파일을 영구적으로 제거합니다.
- 향후 유지 관리 작업이 자동으로 실행되도록 **예측 최적화(predictive optimization)**를 활성화합니다.

이 연습에서는 보존 설정을 구성하고, 레코드 삭제를 시뮬레이션하며, `VACUUM` 작업을 실행합니다.

### 작업 3.1 — 보존 속성 구성

`vehicle_telemetry`에 다음 Delta Lake 테이블 속성을 설정하세요:

- `delta.logRetentionDuration` = `interval 14 days`
- `delta.deletedFileRetentionDuration` = `interval 14 days`

> 🤖 **Genie Code 팁:** 다음과 같이 질문해 보세요:
> *"Databricks에서 Unity Catalog 테이블에 Delta Lake 보존 기간 속성을 어떻게 설정하나요?"*

**힌트:** `ALTER TABLE ... SET TBLPROPERTIES (...)`를 사용하세요.

In [ ]:
%sql
-- TODO: automotive_catalog.governance_lab.vehicle_telemetry에 대해
-- logRetentionDuration과 deletedFileRetentionDuration을 14일로 설정

### 작업 3.2 — GDPR 삭제 요청 시뮬레이션

한 고객이 잊혀질 권리(right-to-erasure) 요청을 제출했습니다. `vehicle_telemetry` 테이블에서 차량 `VH-003`의 모든 텔레메트리 레코드를 삭제하세요.

삭제 후, 테이블을 조회하여 레코드가 사라졌는지 확인하세요.

> 🤖 **Genie Code 팁:** 다음과 같이 질문해 보세요:
> *"Databricks SQL에서 Delta Lake 테이블의 특정 행을 어떻게 삭제하나요?"*

In [ ]:
%sql
-- TODO: vehicle_telemetry에서 vehicle_id = 'VH-003'인 모든 행 삭제

-- TODO: 테이블을 조회하여 삭제 확인

### 작업 3.3 — VACUUM을 실행하여 삭제된 데이터 파일 정리

삭제된 행은 `VACUUM`이 실행되기 전까지 기본 Parquet 파일에 여전히 존재합니다. `vehicle_telemetry`에 `VACUUM`을 실행하여 현재 또는 최근 테이블 버전에서 더 이상 참조되지 않는 파일을 영구적으로 제거하세요.

기본 보존 기간인 **168시간(7일)**을 사용하세요. 이는 안전하고, Serverless 컴퓨팅에서 작동하며, 실습 환경에 적합합니다.

> 🤖 **Genie Code 팁:** 다음과 같이 질문해 보세요:
> *"Databricks에서 Delta Lake 테이블에 VACUUM을 어떻게 실행하며, RETAIN 옵션은 무엇을 하나요?"*

**참고:** Serverless 컴퓨팅에서는 최소 보존 기간을 재정의하는 것이 지원되지 않습니다. 항상 최소 168시간 이상의 `RETAIN` 값을 지정하거나, 기본값을 사용하려면 `RETAIN`을 완전히 생략하세요.

In [ ]:
%sql
-- TODO: automotive_catalog.governance_lab.vehicle_telemetry에 VACUUM 실행
-- RETAIN 168 HOURS(7일)를 사용하거나 기본값을 사용하려면 RETAIN 생략

### 작업 3.4 — 예측 최적화 활성화

향후 `VACUUM` 및 `OPTIMIZE` 유지 관리가 수동 스케줄링 없이 자동으로 실행되도록 `governance_lab` 스키마에서 **예측 최적화(predictive optimization)**를 활성화하세요.

> 🤖 **Genie Code 팁:** 다음과 같이 질문해 보세요:
> *"Databricks에서 Unity Catalog 스키마에 예측 최적화를 어떻게 활성화하나요?"*

**힌트:** `ALTER SCHEMA ... ENABLE PREDICTIVE OPTIMIZATION`을 사용하세요.

In [ ]:
%sql
-- TODO: automotive_catalog.governance_lab에 예측 최적화 활성화

## 연습 4: 데이터 계보를 프로그래밍 방식으로 조회

> 📋 **이 연습을 실행하기 전에**: 아직 수행하지 않았다면 여기서 잠시 멈추고, 실습 설정 지침의 **Catalog Explorer 계보 단계**를 따라 UI에서 계보를 시각적으로 탐색하세요. 그런 다음 여기로 돌아와 SQL을 사용해 계보 데이터를 조회하세요.

Unity Catalog는 모든 읽기 및 쓰기 이벤트를 캡처하여 `system.access.table_lineage` 시스템 테이블에 저장합니다. 이를 통해 *"이번 주에 우리 automotive 카탈로그에서 가장 많이 조회된 테이블은 무엇인가?"*와 같은 질문에 프로그래밍 방식으로 답할 수 있습니다.

### 작업 4.1 — 카탈로그에서 최근 접근된 테이블 찾기

`system.access.table_lineage`를 조회하여 `automotive_catalog`와 관련된 지난 7일 이내의 모든 **쓰기 이벤트**(소스 테이블)를 찾으세요. 테이블 이름과 고유 이벤트 수를 반환하고, 가장 활동이 많은 테이블 순으로 정렬하세요.

> 🤖 **Genie Code 팁:** 다음과 같이 질문해 보세요:
> *"Databricks에서 최근 접근된 Unity Catalog 테이블을 찾기 위해 system.access.table_lineage를 어떻게 조회하나요?"*

**힌트:** `source_table_full_name LIKE 'automotive_catalog%'`로 필터링하고 테이블 이름으로 그룹화하세요.

In [ ]:
%sql
-- TODO: 지난 7일 동안 접근된 automotive_catalog의 테이블을 system.access.table_lineage에서 조회
-- 반환: source_table_full_name, 이벤트 수
-- 가장 활동이 많은 순으로 정렬

### 작업 4.2 — vehicle_telemetry의 이력 보기

`DESCRIBE HISTORY`를 사용하여 `vehicle_telemetry`의 전체 Delta Lake 작업 이력을 확인하세요. 삭제가 발생한 버전(`DELETE` 작업)을 식별하세요.

> 🤖 **Genie Code 팁:** 다음과 같이 질문해 보세요:
> *"Databricks SQL에서 Delta Lake 테이블의 전체 버전 이력을 어떻게 확인하나요?"*

In [ ]:
%sql
-- TODO: automotive_catalog.governance_lab.vehicle_telemetry의 이력 설명

## 연습 5: 감사 로깅(Audit Logging)

AutoSphere의 컴플라이언스 팀은 지난 30일 동안 승인된 사용자만 고객 데이터에 접근했음을 규제 기관에 증명해야 합니다. 또한 Unity Catalog 메타스토어에서 권한이 수정될 때마다 알림을 받고자 합니다.

이 두 가지 요구 사항을 모두 충족하기 위해 `system.access.audit` 시스템 테이블을 사용합니다.

### 작업 5.1 — customer_registrations 테이블에 접근한 사용자 찾기

`system.access.audit`를 조회하여 지난 30일 동안 `automotive_catalog.governance_lab.customer_registrations`에 접근한 모든 사용자를 나열하세요. 각 접근의 작업 이름과 시간을 포함하고, 가장 최근 순으로 정렬하세요.

> 🤖 **Genie Code 팁:** 다음과 같이 질문해 보세요:
> *"특정 Unity Catalog 테이블에 누가 접근했는지 찾기 위해 Databricks 감사 로그 시스템 테이블을 어떻게 조회하나요?"*

**힌트:** `request_params.full_name_arg`와 `action_name IN ('getTable', 'createTable')`로 필터링하세요. 사용자 컬럼에는 `user_identity.email`을 사용하세요.

In [ ]:
%sql
-- TODO: automotive_catalog.governance_lab.customer_registrations에 대한 접근 이벤트를 system.access.audit에서 조회
-- 반환: 사용자 이메일, action_name, event_time
-- 지난 30일로 필터링, event_time DESC로 정렬

### 작업 5.2 — 카탈로그 전반의 권한 변경 감지

`system.access.audit`를 조회하여 지난 30일 동안 `automotive_catalog`의 보안 가능 객체(securable)에 대한 모든 **권한 변경 이벤트**(`updatePermissions` 작업)를 나열하세요. 변경을 수행한 사람과 영향을 받은 객체를 포함하세요.

> 🤖 **Genie Code 팁:** 다음과 같이 질문해 보세요:
> *"특정 Unity Catalog 카탈로그에 대한 모든 권한 변경을 Databricks 감사 로그에서 어떻게 찾나요?"*

**힌트:** `service_name = 'unityCatalog'`, `action_name = 'updatePermissions'`, `request_params.securable_full_name LIKE 'automotive_catalog%'`로 필터링하세요.

In [ ]:
%sql
-- TODO: automotive_catalog 객체에 대한 권한 변경 이벤트를 system.access.audit에서 조회
-- 반환: event_time, 사용자 이메일, securable_type, securable_full_name, changes
-- 지난 30일로 필터링, event_time DESC로 정렬

### 작업 5.3 — 사용자별 감사 활동 요약

지난 7일 동안 Unity Catalog 작업에 대한 사용자별 총 감사 이벤트 수를 집계하는 요약 쿼리를 작성하세요. 이를 통해 컴플라이언스 팀은 플랫폼에서 누가 가장 활발히 활동했는지 빠르게 파악할 수 있습니다.

> 🤖 **Genie Code 팁:** 다음과 같이 질문해 보세요:
> *"Unity Catalog 활동에 대해 사용자별 이벤트 수를 집계하기 위해 Databricks 감사 로그를 어떻게 집계하나요?"*

In [ ]:
%sql
-- TODO: system.access.audit에서 사용자별 총 감사 이벤트 수 집계
-- service_name = 'unityCatalog'와 지난 7일로 필터링
-- 반환: 사용자 이메일, 이벤트 수
-- 가장 활동이 많은 사용자 순으로 정렬